In [ ]:
# -*- coding: utf-8 -*-
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or
# implied.
# See the License for the specific language governing permissions and
# limitations under the License.


# Imports

In [ ]:
import pandas as pd
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.model_selection import StratifiedKFold, cross_val_score, RandomizedSearchCV, GridSearchCV
import numpy as np
from sklearn.decomposition import TruncatedSVD
import os
import wandb
from scipy.stats import uniform, loguniform, randint
import sys
sys.path.append(os.path.join(os.getcwd(), '..'))
from util.preprocessing import TweetPreprocessor 
import joblib
import time

# Consts

In [ ]:
MODEL_DIR = os.path.join(os.getcwd(), 'models')
DATASETS_DIR = os.path.join(os.getcwd(), 'datasets')

# Dataset

In [ ]:
train_df = pd.read_csv(os.path.join(DATASETS_DIR, "one_tweet_dataset_train.csv"))
val_df = pd.read_csv(os.path.join(DATASETS_DIR, "one_tweet_dataset_val.csv"))
train_df = pd.concat([train_df, val_df], ignore_index=True)

In [ ]:
x_train, y_train = train_df["text"], train_df["gender_label"]

# Initiate pipeline

In [ ]:
pipeline = Pipeline([
    ("preprocessor", TweetPreprocessor()),
    ("features", FeatureUnion([
        ("tfidf_word", TfidfVectorizer(analyzer="word")), # type: ignore
        ("tfidf_char", TfidfVectorizer(analyzer="char")),
    ])),
    ("svd", TruncatedSVD(random_state=int(os.getenv("RANDOM_SEED", 880055535)))), # type: ignore
    ("clf", LinearSVC(random_state=int(os.getenv("RANDOM_SEED", 880055535)))), # type: ignore
])

# Init wandb

In [ ]:
wandbToken = os.getenv("WANDB_TOKEN")
if not wandbToken:
    raise ValueError("Please set the WANDB_TOKEN environment variable to log results to Weights & Biases.")
wandb.login(key=wandbToken)

# Randomized search

In [ ]:
ngram_ranges_word = [(1, 2), (1, 3), (2, 3)]
ngram_ranges_char = [(2, 4), (3, 5), (4, 6), (2, 5)]

rnd_params = {
    "features__tfidf_word__use_idf": [True, False],
    "features__tfidf_word__sublinear_tf": [True, False],
    "features__tfidf_word__norm": ["l1", "l2"],
    "features__tfidf_word__max_df": uniform(0.6, 0.4),
    "features__tfidf_word__min_df": uniform(0.001, 0.4),
    "features__tfidf_word__max_features": randint(5000, 120000),
    "features__tfidf_word__ngram_range": ngram_ranges_word,

    "features__tfidf_char__use_idf": [True, False],
    "features__tfidf_char__sublinear_tf": [True, False],
    "features__tfidf_char__norm": ["l1", "l2"],
    "features__tfidf_char__max_df": uniform(0.6, 0.4),
    "features__tfidf_char__min_df": uniform(0.001, 0.4),
    "features__tfidf_char__max_features": randint(5000, 120000),
    "features__tfidf_char__ngram_range": ngram_ranges_char,

    "svd__n_components": randint(10, 400),

    "clf__C": loguniform(1e-2, 1e2),
}

In [ ]:
rnd_run = wandb.init(project="who-wrote-it-nlp", name="hyperparam_search", entity="who-wrote-it-nlp", job_type="hyperparam_search_rnd",group="random_search")

rnd_search = RandomizedSearchCV(
    pipeline, 
    rnd_params, 
    n_iter=500,
    cv=StratifiedKFold(n_splits=4, shuffle=True, random_state=int(os.getenv("RANDOM_SEED", 880055535))),
    scoring="f1_macro",
    n_jobs=20,
    verbose=10,
    random_state=int(os.getenv("RANDOM_SEED", 880055535))
)


In [ ]:
rnd_search.fit(x_train, y_train)

In [ ]:
for i, row in pd.DataFrame(rnd_search.cv_results_).iterrows():
    params = row["params"]
    score = row["mean_test_score"]
    wandb.log({
        **params,
        "score": score,
        "std" : row["std_test_score"]
    })

best_rnd = rnd_search.best_params_
best_rnd_score = rnd_search.best_score_
best_rnd_model = rnd_search.best_estimator_
wandb.log({
    "best_score": best_rnd_score,
    "best_params": best_rnd
})

rnd_model_path = os.path.join(MODEL_DIR, "best_rnd_model.joblib")
joblib.dump(best_rnd_model, rnd_model_path)

artifact = wandb.Artifact("best_rnd_model", type="model")
artifact.add_file(rnd_model_path)
wandb.log_artifact(artifact)

rnd_run.finish()

# Grid search

In [ ]:
def make_range(value, pct=0.05, cast_int=False):
    factors = [1 - pct, 1, 1 + pct]

    candidates = [value * f for f in factors]

    if cast_int:
        candidates = [int(round(c)) for c in candidates]

    uniq = []
    for c in candidates:
        if c not in uniq:
            uniq.append(c)

    return uniq

grid_params = {
    "features__tfidf_word__use_idf": [best_rnd["features__tfidf_word__use_idf"]],
    "features__tfidf_word__sublinear_tf": [best_rnd["features__tfidf_word__sublinear_tf"]],
    "features__tfidf_word__norm": [best_rnd["features__tfidf_word__norm"]],
    "features__tfidf_word__ngram_range": [best_rnd["features__tfidf_word__ngram_range"]],
    "features__tfidf_word__max_df": make_range(best_rnd["features__tfidf_word__max_df"]),
    "features__tfidf_word__min_df": make_range(best_rnd["features__tfidf_word__min_df"]),
    "features__tfidf_word__max_features": make_range(best_rnd["features__tfidf_word__max_features"], cast_int=True),

    "features__tfidf_char__use_idf": [best_rnd["features__tfidf_char__use_idf"]],
    "features__tfidf_char__sublinear_tf": [best_rnd["features__tfidf_char__sublinear_tf"]],
    "features__tfidf_char__norm": [best_rnd["features__tfidf_char__norm"]],
    "features__tfidf_char__ngram_range": [best_rnd["features__tfidf_char__ngram_range"]],
    "features__tfidf_char__max_df": make_range(best_rnd["features__tfidf_char__max_df"]),
    "features__tfidf_char__min_df": make_range(best_rnd["features__tfidf_char__min_df"]),
    "features__tfidf_char__max_features": make_range(best_rnd["features__tfidf_char__max_features"], cast_int=True),

    "svd__n_components": [best_rnd["svd__n_components"]],

    "clf__C": make_range(best_rnd["clf__C"]),
}

In [ ]:
grid_run = wandb.init(project="who-wrote-it-nlp", name="hyperparam_search", entity="who-wrote-it-nlp", job_type="hyperparam_search_grid",group="grid_search")
grid_search = GridSearchCV(
    pipeline, 
    grid_params,
    cv=StratifiedKFold(n_splits=4, shuffle=True, random_state=int(os.getenv("RANDOM_SEED", 880055535))),
    scoring="f1_macro",
    n_jobs=20,
    verbose=10,
)

In [ ]:
grid_search.fit(x_train, y_train)

In [ ]:
for i, row in pd.DataFrame(grid_search.cv_results_).iterrows():
    params = row["params"]
    score = row["mean_test_score"]
    wandb.log({
        **params,
        "score": score,
        "std" : row["std_test_score"]
    })

best_grid = grid_search.best_params_
best_grid_score = grid_search.best_score_
best_grid_model = grid_search.best_estimator_
wandb.log({
    "best_score": best_grid_score,
    "best_params": best_grid
})

grid_model_path = os.path.join(MODEL_DIR, "best_grid_model.joblib")
joblib.dump(best_grid_model, grid_model_path)
artifact = wandb.Artifact("best_grid_model", type="model")
artifact.add_file(grid_model_path)
wandb.log_artifact(artifact)

grid_run.finish()